# Lab 8: Exception Handling and Logging

### 💡 What is Exception Handling?
Imagine you're following a recipe to bake a cake 🎂, but you realize you're out of eggs. If you just stop and stare at the bowl, your 'process' crashes! **Exception Handling** is like having a backup plan: "If I don't have eggs, I will use applesauce instead." In coding, it prevents your program from crashing when something unexpected happens (like a missing file).

### 📝 Why Logging Matters?
Logging is like keeping a **diary** 📖 for your code. Instead of just seeing that the cake failed, the log tells you: *"At 10:05 AM, I looked for eggs and couldn't find them."* It helps you retrace your steps to find out exactly where and why things went wrong.

### 🛠️ What we will learn:
- How to use `try/except` to catch errors.
- How to handle missing files and messy JSON data.
- How to create a professional logging system to track your code's journey.
- **The Goal:** Build programs that are 'bulletproof' and don't just quit when they hit a tiny bump in the road! 🛡️

## Objectives
By the end of this lab, you will be able to:
* Implement proper **exception handling** using `try/except` blocks.
* Handle specific exceptions like `FileNotFoundError` and `JSONDecodeError`.
* Use `finally` blocks for resource cleanup.
* Configure Python's `logging` module for console and file output.
* Create robust applications that handle errors gracefully.

## Environment Setup
Google Colab provides a clean slate. We'll start by importing the necessary built-in modules and ensuring we have a clean directory for our logs.

In [1]:
import os
import json
import logging
import logging.handlers
import sys
from datetime import datetime
from pathlib import Path

# Create a directory for our logs if it doesn't exist
if not os.path.exists('logs'):
    os.makedirs('logs')
    print("✅ Created 'logs' directory.")
else:
    print("ℹ️ 'logs' directory already exists.")

✅ Created 'logs' directory.


## Task 1: Basic Exception Handling for File Operations

In this task, we will learn how to read files safely. We'll compare a 'risky' way to read files versus a 'safe' way using `try/except/finally`.

In [2]:
# Define the file reading functions

def read_file_basic(filename):
    # This is UNSAFE. If the file doesn't exist, the program crashes immediately.
    file = open(filename, 'r', encoding='utf-8')
    content = file.read()
    file.close()
    return content

def read_file_with_exceptions(filename):
    # This is SAFE. We anticipate potential problems.
    file = None
    try:
        print(f"🔍 Attempting to open file: {filename}")
        file = open(filename, 'r', encoding='utf-8')
        content = file.read()
        print(f"✅ Successfully read {len(content)} characters.")
        return content
    except FileNotFoundError:
        print(f"❌ Error: The file '{filename}' was not found.")
        return None
    except PermissionError:
        print(f"❌ Error: Permission denied for '{filename}'.")
        return None
    except IOError as e:
        print(f"❌ Error: An I/O error occurred: {e}")
        return None
    finally:
        # This runs NO MATTER WHAT. Perfect for closing files.
        if file and not file.closed:
            file.close()
            print(f"🔒 File '{filename}' has been properly closed.")

# --- Execution ---
print("--- Testing Safe Reader ---")
# We'll create a dummy file first to test success
with open("dummy.txt", "w") as f: f.write("Hello Colab!")

read_file_with_exceptions("dummy.txt")
print("\n--- Testing with Missing File ---")
read_file_with_exceptions("imaginary_file.txt")

--- Testing Safe Reader ---
🔍 Attempting to open file: dummy.txt
✅ Successfully read 12 characters.
🔒 File 'dummy.txt' has been properly closed.

--- Testing with Missing File ---
🔍 Attempting to open file: imaginary_file.txt
❌ Error: The file 'imaginary_file.txt' was not found.


### Subtask 1.2: Enhanced File Operations with Context Managers
Python's `with` statement (Context Manager) is the gold standard. It automatically closes the file for you, even if an error occurs inside the block!

In [3]:
def write_file_safely(filename, content):
    try:
        # Using 'with' handles file closing automatically!
        with open(filename, 'w', encoding='utf-8') as file:
            file.write(content)
        print(f"📝 Saved data to {filename}")
        return True
    except Exception as e:
        print(f"💥 Unexpected error writing to {filename}: {e}")
        return False

def read_file_with_context_manager(filename):
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            content = file.read()
        return content
    except FileNotFoundError:
        print(f"❓ File '{filename}' not found.")
        return None

# Create test files
write_file_safely("test_data.txt", "Sample data content.")
write_file_safely("special_chars.txt", "Special characters: áéíóú ñ")

# Read back
print(f"Read test_data: {read_file_with_context_manager('test_data.txt')}")

📝 Saved data to test_data.txt
📝 Saved data to special_chars.txt
Read test_data: Sample data content.


## Task 2: JSON Exception Handling and Data Validation

JSON is everywhere! But it's easy to break (forgetting a comma is a classic error). We'll learn how to catch formatting errors and validate that the data contains what we expect.

In [4]:
def process_json_demo(filename, data_to_write, is_malformed=False):
    # 1. Write the file
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            if is_malformed:
                f.write(data_to_write) # Writing raw broken string
            else:
                json.dump(data_to_write, f, indent=2)
        print(f"💾 Created {filename}")
    except Exception as e:
        print(f"Error creating file: {e}")

    # 2. Try to read it back
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"✅ Parsed JSON from {filename}: {data}")
    except json.JSONDecodeError as e:
        print(f"⚠️ JSON Error in {filename}: {e.msg} at line {e.lineno}")

# Testing
valid_obj = {"name": "Alice", "age": 30}
broken_str = '{"name": "Bob", "age": 25 ' # Missing closing brace

process_json_demo("valid.json", valid_obj)
process_json_demo("broken.json", broken_str, is_malformed=True)

💾 Created valid.json
✅ Parsed JSON from valid.json: {'name': 'Alice', 'age': 30}
💾 Created broken.json
⚠️ JSON Error in broken.json: Expecting ',' delimiter at line 1


### Subtask 2.2: Advanced JSON with Custom Exceptions
Sometimes `ValueError` isn't descriptive enough. We can create our own error types to make our code read like a story.

In [5]:
class JSONFileError(Exception): pass
class JSONValidationError(JSONFileError): pass

class JSONProcessor:
    def __init__(self):
        self.stats = {"success": 0, "fail": 0}

    def load_and_validate(self, filename, required_fields):
        try:
            if not os.path.exists(filename):
                raise FileNotFoundError(f"{filename} is missing!")

            with open(filename, 'r', encoding='utf-8') as f:
                data = json.load(f)

            # Custom validation logic
            for field in required_fields:
                if field not in data:
                    raise JSONValidationError(f"Missing field: {field}")

            self.stats["success"] += 1
            return data

        except (FileNotFoundError, json.JSONDecodeError, JSONValidationError) as e:
            print(f"❌ Processing Failed for {filename}: {e}")
            self.stats["fail"] += 1
            return None

# Usage
processor = JSONProcessor()
with open("user.json", "w") as f: json.dump({"id": 1, "name": "Charlie"}, f)

print("Checking user.json for 'email'...")
processor.load_and_validate("user.json", ["id", "name", "email"])
print(f"Current Stats: {processor.stats}")

Checking user.json for 'email'...
❌ Processing Failed for user.json: Missing field: email
Current Stats: {'success': 0, 'fail': 1}


## Task 3: Comprehensive Logging Configuration

Now, we move beyond `print()` statements. Logging allows us to save errors to files and filter them by importance (DEBUG, INFO, WARNING, ERROR, CRITICAL).

In [6]:
# Basic Logging Setup
# IMPORTANT: In Colab, we clear handlers to avoid duplicate logs when re-running cells.
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('application.log', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger("BasicApp")

logger.info("🚀 Application started.")
logger.debug("This is a debug message - usually for developers.")
logger.warning("This is a warning - something feels fishy!")
logger.error("This is an error - something actually broke!")

2026-04-13 19:20:50,670 - INFO - 🚀 Application started.
2026-04-13 19:20:50,674 - DEBUG - This is a debug message - usually for developers.
2026-04-13 19:20:50,679 - WARNING - This is a warning - something feels fishy!
2026-04-13 19:20:50,682 - ERROR - This is an error - something actually broke!


### Subtask 3.2: Advanced Logging (Rotating & Multi-handler)
In production, log files can grow huge! A **Rotating Handler** limits the file size and creates backups automatically.

In [7]:
class AdvancedProcessor:
    def __init__(self):
        self.logger = logging.getLogger("AdvancedApp")
        self.logger.setLevel(logging.DEBUG)
        self.logger.handlers.clear()

        # 1. Error Log (Only Errors go here)
        err_handler = logging.FileHandler('logs/errors.log')
        err_handler.setLevel(logging.ERROR)
        err_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))

        # 2. Rotating Handler (Main App activity)
        rot_handler = logging.handlers.RotatingFileHandler(
            'logs/app_rotating.log', maxBytes=2000, backupCount=3
        )
        rot_handler.setLevel(logging.INFO)

        self.logger.addHandler(err_handler)
        self.logger.addHandler(rot_handler)
        self.logger.addHandler(logging.StreamHandler(sys.stdout))

    def run_task(self, name):
        self.logger.info(f"Running task: {name}")
        if "fail" in name:
            self.logger.error(f"Task {name} encountered a critical failure!")

adv_proc = AdvancedProcessor()
adv_proc.run_task("Initialize System")
adv_proc.run_task("Simulate fail-scenario")

print("\n--- Checking Log Files ---")
!ls logs

Running task: Initialize System
2026-04-13 19:20:51,440 - INFO - Running task: Initialize System
Running task: Simulate fail-scenario
2026-04-13 19:20:51,445 - INFO - Running task: Simulate fail-scenario
Task Simulate fail-scenario encountered a critical failure!
2026-04-13 19:20:51,455 - ERROR - Task Simulate fail-scenario encountered a critical failure!

--- Checking Log Files ---
app_rotating.log  errors.log


## Verification Section
Let's prove the files were created and contains our logs.

In [8]:
print("1. Checking for log files...")
files = os.listdir('logs')
print(f"Files in /logs: {files}")

print("\n2. Previewing 'errors.log':")
if os.path.exists('logs/errors.log'):
    with open('logs/errors.log', 'r') as f:
        print(f.read())

print("\n3. Checking data files:")
!ls *.json

1. Checking for log files...
Files in /logs: ['errors.log', 'app_rotating.log']

2. Previewing 'errors.log':
2026-04-13 19:20:51,455 - ERROR - Task Simulate fail-scenario encountered a critical failure!


3. Checking data files:
broken.json  user.json	valid.json


## Troubleshooting
*   **Duplicate Logs:** If you see the same log message 5 times, it's because you re-ran the cell. Use `logger.handlers.clear()` before adding new handlers.
*   **Missing Files:** Remember that Colab deletes your files if the runtime disconnects. Always re-run the "Setup" and "Task" cells if you come back after a break.
*   **Encoding Errors:** Always use `encoding='utf-8'` when opening files to avoid issues with special characters (like emojis or accents).

## Conclusion
You've successfully built a resilient Python application!

### Key Takeaways:
1.  **Try/Except** is your safety net.
2.  **Finally/With** ensures your resources (like files) are never left hanging.
3.  **Logging** provides a professional way to monitor code without cluttering the screen with `print` statements.
4.  **Custom Exceptions** make your code easier to debug and maintain.

### What I Learned in this Lab (Learner Perspective):
*"I learned that errors aren't just 'bad'—they are events I can plan for. By using logging, I can see exactly what my script was doing at 2 AM without being there to watch it. I also realized that using context managers (`with`) makes my code much cleaner and safer."*